# Real-Time CV Object Tracking - YOLO26 Local Live

Run this end-to-end to analyze **live video** with the YOLO26-m detector on your own machine.

**What you get, top-to-bottom:**
- Setup: dependency check, then load the YOLO26-m detector.
- Camera picker: pick ONE source (curated public HLS, custom URL, or your own uploaded MP4/MKV).
- Sections 1-6: single-frame check, footfall time series, anomalies, dwell tracking, appearance re-ID.
- Section 7: **the live dashboard**, embedded inline, showing the picked source full-width.
- Sections 8-10: business score, multi-source ranking, accuracy calibration.

The dashboard supports 10 analysis layers on the tile itself: paths / pose / gestures / body / faces / line / loiter / parking / plates / heat. Only one layer runs at a time (operator constraint).

**Supported source types (public streaming and local file only):**
- Direct HLS (`.m3u8`) - Skylinewebcams, custom
- webcamera24.com pages
- skylinewebcams.com pages
- Your own uploaded MP4/MKV files (upload button in the dashboard header)

**Prereqs:** Python 3.10+, `pip install -r src/requirements.txt`, run this notebook from the repo root.

### Public HLS stream reality

Public live cameras are exposed as HLS (`.m3u8`) manifests. Most are reachable from any network with an open connection to the origin CDN; a few (municipal ones behind CDNs) require specific headers - the code handles those via a small pre-fetch helper. If a picked camera fails to open, try another from the catalog or upload a local MP4/MKV instead.

## 0. Setup

In [1]:
# Dependency check: verify every library the notebook needs is installed,
# install any that are missing, then print the installed versions.
# Safe to re-run - subsequent runs are just a version dump.
import importlib, subprocess, sys

# (import_name, pip_name). Pinned only where a min version matters.
REQUIREMENTS = [
    ('cv2',            'opencv-python-headless'),
    ('numpy',          'numpy'),
    ('pandas',         'pandas'),
    ('matplotlib',     'matplotlib'),
    ('PIL',            'Pillow'),
    ('ultralytics',    'ultralytics'),
    ('ipywidgets',     'ipywidgets>=8'),
    ('urllib3',        'urllib3'),
]

def _pip_install(spec: str) -> None:
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', spec],
        stdout=sys.stdout, stderr=sys.stderr,
    )

def _version(mod) -> str:
    return getattr(mod, '__version__', getattr(mod, 'VERSION', 'unknown'))

print(f'{"import":18} {"pip package":24} status')
print('-' * 66)
missing = []
for import_name, pip_spec in REQUIREMENTS:
    try:
        m = importlib.import_module(import_name)
        print(f'{import_name:18} {pip_spec:24} OK  v{_version(m)}')
    except ImportError:
        print(f'{import_name:18} {pip_spec:24} MISSING -> installing')
        missing.append((import_name, pip_spec))

for import_name, pip_spec in missing:
    _pip_install(pip_spec)
    importlib.invalidate_caches()
    m = importlib.import_module(import_name)
    print(f'  {import_name:16} installed  v{_version(m)}')

if missing:
    print()
    print('NOTE: some packages were just installed. If the next cell errors '
          'with ModuleNotFoundError, restart the kernel (Kernel -> Restart) '
          'and re-run from the top so Python picks up the new installs.')


import             pip package              status
------------------------------------------------------------------
cv2                opencv-python-headless   OK  v4.13.0
numpy              numpy                    OK  v2.3.5
pandas             pandas                   OK  v2.3.3
matplotlib         matplotlib               OK  v3.10.6
PIL                Pillow                   OK  v12.0.0
ultralytics        ultralytics              OK  v8.4.67
ipywidgets         ipywidgets>=8            OK  v8.1.7
urllib3            urllib3                  OK  v2.5.0


### 0b. Model weights (auto-downloaded on first run)

This project does not ship model weights in the git repo — nothing binary lives on GitHub. The cell below downloads every optional weight the pipeline may need directly onto whatever machine is running the notebook, on the first run only. Subsequent runs skip files that already exist. Nothing is written to the repo's tracked tree.

What gets fetched here:

| File | Layer | Source | License |
|---|---|---|---|
| `yolov8n-plate.pt` | LPR stage 1 (plate detector) | HuggingFace `Koushim/yolov8-license-plate-detection` | MIT |
| `plate_ocr_global.onnx` | LPR stage 2 (OCR) | GitHub `ankandrew/fast-plate-ocr` releases | MIT |
| `models/FSRCNN_x4.pb` | Plate 4× super-resolution | GitHub `Saafke/FSRCNN_Tensorflow` | Apache-2.0 |
| `src/data/face_detection_yunet_2023mar.onnx` | Face detection layer | GitHub `opencv/opencv_zoo` | Apache-2.0 |

`yolo26m.pt` and `yolov8n-pose.pt` are auto-downloaded by `ultralytics` in the next cell (cell 4) — this cell doesn't touch them.

OSNet Re-ID ONNX is **not** downloaded (no permissive direct URL). The Re-ID layer falls back to an HSV colour histogram in its absence — the default already-working behaviour.

If any download fails (network glitch, upstream URL changed), the cell prints a clear error and moves on. Layers that depend on a failed weight will show "model not loaded" but every other layer still works.


In [ ]:
# Auto-fetch every non-ultralytics model weight the pipeline may need.
# Runs on first notebook execution; skips any file already on disk.
# NOTHING is checked into git - .gitignore excludes all *.pt, *.onnx, *.pb
# and *_openvino_model/ paths, so every download lands OUTSIDE version control.
import os, ssl, urllib.request
from pathlib import Path

# Locate the repo root regardless of whether the notebook is run from the
# repo root (default) or from inside src/.
_REPO = Path.cwd() if (Path.cwd() / 'src' / 'app').is_dir() else Path.cwd().parent

# Each row: (dest_relative_to_repo_root, primary_url, [fallback_urls...], description).
# Fallback URLs are tried in order if the primary returns HTTP >=400 or times out.
_FETCH_MANIFEST = [
    (
        'yolov8n-plate.pt',
        'https://huggingface.co/Koushim/yolov8-license-plate-detection/resolve/main/best.pt',
        ['https://huggingface.co/Koushim/yolov8-license-plate-detection/resolve/main/yolov8n-plate.pt'],
        'LPR stage 1 (Koushim yolov8n plate detector, MIT)',
    ),
    (
        'plate_ocr_global.onnx',
        'https://github.com/ankandrew/fast-plate-ocr/releases/download/cct-xs-v1-global/cct-xs-v1-global.onnx',
        ['https://github.com/ankandrew/fast-plate-ocr/releases/download/arg-plates/plate_ocr_global.onnx'],
        'LPR stage 2 (fast-plate-ocr cct-xs-v1-global, MIT)',
    ),
    (
        'models/FSRCNN_x4.pb',
        'https://raw.githubusercontent.com/Saafke/FSRCNN_Tensorflow/master/models/FSRCNN_x4.pb',
        [],
        'Plate 4x super-res (Saafke FSRCNN, Apache-2.0)',
    ),
    (
        'src/data/face_detection_yunet_2023mar.onnx',
        'https://raw.githubusercontent.com/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx',
        [],
        'Face detection (opencv_zoo YuNet, Apache-2.0)',
    ),
]

_ctx = ssl.create_default_context()  # respect the system CA bundle

def _download(url: str, dest: Path, timeout: int = 90) -> int:
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (real_time_cv notebook)'})
    with urllib.request.urlopen(req, timeout=timeout, context=_ctx) as r:
        data = r.read()
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(data)
    return len(data)

_ok, _skipped, _failed = [], [], []
for rel, primary, fallbacks, desc in _FETCH_MANIFEST:
    dest = _REPO / rel
    if dest.exists() and dest.stat().st_size > 1024:
        _skipped.append(rel)
        print(f'  skip  {rel}  (already present, {dest.stat().st_size/1024/1024:.1f} MB)')
        continue
    urls = [primary] + list(fallbacks)
    last_err = None
    for i, url in enumerate(urls):
        label = 'primary' if i == 0 else f'fallback {i}'
        try:
            print(f'  fetch {rel}  ({label})...')
            n = _download(url, dest)
            print(f'         -> OK {n/1024/1024:.1f} MB')
            _ok.append(rel)
            break
        except Exception as e:
            last_err = f'{type(e).__name__}: {e}'
            print(f'         -> failed ({last_err})')
    else:
        _failed.append((rel, desc, last_err))

print()
print(f'weights: {len(_ok)} downloaded, {len(_skipped)} already present, {len(_failed)} failed')
for rel, desc, err in _failed:
    print(f'  ! {rel}  ({desc})')
    print(f'    last error: {err[:120]}')
if _failed:
    print()
    print('Note: any layer that needs a failed weight will show "model not loaded"')
    print('and be skipped. Every other layer + the notebook itself still work.')
    print('To fetch manually, see the README Model files section for URLs.')

In [2]:
import os, sys, time, datetime as dt
# When yt-dlp is blocked at IP level by YouTube's bot check, resolve_stream
# short-circuits to a screen://primary sentinel and grab_frame reads pixels
# off the operator's primary display via mss/PIL. Opt-in here so the
# notebook works out of the box on operator machines with a browser tab
# already showing the camera.
os.environ.setdefault('SCREEN_CAPTURE_FALLBACK', '1')
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the src/ tree so the `app` package imports regardless of whether the
# notebook is run from the project root (default layout) or from inside src/.
_src_dir = Path.cwd() / 'src' if (Path.cwd() / 'src' / 'app').is_dir() else Path.cwd()
sys.path.append(str(_src_dir))
from app.detect_core import load_model, detect_and_count, grab_frame, resolve_stream, VEHICLE_NAMES
from app.cameras import CAMERAS, active_cameras

# --- Model: the local STRONG reference detector ---------------------------
# YOLO26-x (extra-large) is the accuracy-priority variant of the 2026 YOLO
# generation: mAP50-95 on COCO is 57.5 vs 53.1 for yolo26m, at the cost of
# ~2.4x CPU tick latency. For the notebook (which runs bursts of frames on
# demand rather than a live stream) accuracy wins - the OpenVINO IR at
# src/yolo26x_openvino_model/ still keeps the tick well below one second
# per frame on i5-class laptops.
#
# If you prefer the medium (faster) variant, change to 'yolo26m.pt'; the
# OpenVINO IR ships for both. Both .pt files ship at the repo root; the
# loader resolves the sibling _openvino_model/ automatically.
MODEL_WEIGHTS = 'yolo26x.pt'
DATA_DIR = _src_dir / 'data'; DATA_DIR.mkdir(parents=True, exist_ok=True)
model = load_model(str(_src_dir / MODEL_WEIGHTS))
print('model:', MODEL_WEIGHTS, '(local YOLO26 extra-large)')
print('cameras available:', list(active_cameras().keys()))
print('mode: single-camera live (pick one in the picker below)')

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
detect: OpenVINO engine loaded (C:\Users\OR\Downloads\לימודים\Data Science\Project\Real_Time_CV_Object_Tracking_YOLO26\src\yolo26x_openvino_model) - adapter overlay skipped (torch-only)
model: yolo26x.pt (local YOLO26 extra-large)
cameras available: ['th_sukhumvit', 'th_chaweng_hooters', 'th_nanai_road', 'th_patong_sainamyen', 'th_petchaburi_traffic', 'th_green_mango', 'th_sukhumvit_soi11', 'th_chaweng_pancake', 'th_chaweng_murphys']
mode: single-camera live (pick one in the picker below)


### Camera picker - pick ONE source

The picker lists catalog cameras (public HLS / webcamera24 / skyline / YouTube) plus any uploaded MP4/MKV files (see the dashboard's Upload button after Section 7).

**The next code cell is INTERACTIVE and picks a SINGLE camera only.** It will print a prompt and pause the notebook until you type ONE camera number and press Enter. "Run All" will stop there and wait for you — nothing runs on a random default. Multiple numbers, ranges, or comma-separated lists are rejected: this pipeline is single-source by design. For hands-off runs (CI, cron) set `NOTEBOOK_PICK=<n>` in the environment before starting Jupyter and the prompt is skipped.


In [3]:
from app.cameras import CAMERAS, active_cameras
_cams = active_cameras()
if not _cams:
    print('No cameras defined. Add entries to src/app/cameras.py or upload a file via the dashboard.')
else:
    print(f'{len(_cams)} camera(s) available:')
    for i, (cid, cam) in enumerate(_cams.items(), 1):
        area = cam.get('area', '')
        print(f'  {i:2d}. [{cam.get("kind", "?"):<12}] {cam["name"]}  ({area})')

9 camera(s) available:
   1. [youtube     ] Sukhumvit Rd (Bangkok)  ()
   2. [youtube     ] Chaweng Beach Rd (Koh Samui)  ()
   3. [youtube     ] Nanai Rd (Patong)  ()
   4. [youtube     ] Sainamyen Rd (Patong)  ()
   5. [youtube     ] Petchaburi Rd traffic (Bangkok)  ()
   6. [youtube     ] Soi Green Mango (Chaweng)  ()
   7. [youtube     ] Sukhumvit Soi 11 - El Gaucho (Bangkok)  ()
   8. [youtube     ] Chaweng - Pancake Man (Koh Samui)  ()
   9. [youtube     ] Chaweng - Murphy's Irish Pub (Koh Samui)  ()


In [ ]:
# INTERACTIVE PICKER — this cell PAUSES the notebook and asks you which
# ONE camera to analyse. Type ONE number from the list above and press
# Enter. The picker selects a SINGLE camera only — the notebook runs
# end-to-end on that one source.
#
# For hands-off runs (CI, headless), set the NOTEBOOK_PICK environment
# variable to the camera number BEFORE launching Jupyter and the prompt
# is skipped.
import os

_all = list(active_cameras().items())
if not _all:
    raise RuntimeError("No cameras. Add to cameras.py or upload via dashboard.")

_env_pick = os.environ.get('NOTEBOOK_PICK')
if _env_pick:
    _raw = _env_pick.strip()
    print(f'NOTEBOOK_PICK={_raw} (from environment - skipping the interactive prompt)')
else:
    print(f'{len(_all)} camera(s) available (see list above).')
    _raw = input(f'>>> Pick ONE camera number (1-{len(_all)}) and press Enter: ').strip()

if not _raw:
    raise RuntimeError(
        'No camera picked. Type ONE number in the input prompt above and re-run '
        'this cell, or set NOTEBOOK_PICK=<n> in the environment for hands-off runs.'
    )
# Guard against operators who paste multiple numbers or a range: the
# picker is single-camera only, by design.
if any(sep in _raw for sep in (',', ' ', '-', ';', '/')):
    raise ValueError(
        f'Got {_raw!r}. The picker selects ONE camera only — pass a single '
        f'number between 1 and {len(_all)}. If you want to compare multiple '
        f'cameras, re-run the notebook once per camera.'
    )
try:
    PICK = int(_raw)
except ValueError:
    raise ValueError(f'Not a number: {_raw!r}. Type just the digit shown next to your camera above.')

if not (1 <= PICK <= len(_all)):
    raise IndexError(f'PICK must be between 1 and {len(_all)} (got {PICK})')

SELECTED_CAM_ID, SELECTED_CAM = _all[PICK - 1]
SELECTED_CAMS = [SELECTED_CAM_ID]  # list-of-one for downstream cells; SINGLE camera only
SELECTED_CAMS_APPLIED = True
print(f'Selected #{PICK}: {SELECTED_CAM_ID} - {SELECTED_CAM["name"]}')

### Picked camera

A quick one-line confirmation of which camera the rest of the notebook will analyze.

In [5]:
if not globals().get('SELECTED_CAMS_APPLIED'):
    raise RuntimeError('Run the picker cell above first.')
print(f'Notebook will analyze: {SELECTED_CAM_ID} - {SELECTED_CAM["name"]}')

Notebook will analyze: th_sukhumvit - Sukhumvit Rd (Bangkok)


## 1. Pick a camera

The catalog lives in `app/cameras.py`. Each entry declares a `kind` (`hls`, `skyline`, `webcamera24`, or `local_file`). `resolve_stream(cam)` turns the entry into an openable HLS URL and caches the result until any embedded token expires:

- `hls` - the URL is used directly.
- `skyline` - the tokenized `hd-auth.skylinewebcams.com` playlist, scraped live.
- `webcamera24` - the embedded stream on the webcamera24 page.
- `local_file` - a path to an MP4/MKV/MOV/AVI/WEBM you uploaded via the dashboard.

`SELECTED_CAMS[0]` (a list-of-one from the numeric picker) is the one this section inspects.

In [6]:
if not globals().get('SELECTED_CAMS_APPLIED'):
    class _ApplyFirst(Exception):
        def _render_traceback_(self):
            return ['PAUSED: run the interactive picker cell above and pick ONE camera number first.']
    raise _ApplyFirst()
# Read the operator's first choice from the verify cell above so the
# rest of this notebook analyses whatever camera they actually picked
# (instead of a hard-coded default that ignored their choice).
CAM_ID = SELECTED_CAMS[0]
cam = CAMERAS[CAM_ID]
stream_url = resolve_stream(cam)   # handles hls / skyline / webcamera24
print(cam['name'], '->', stream_url)


ERROR: [youtube] Q71sLS8h9a4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] Q71sLS8h9a4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
ERROR: [youtube] Q71sLS8h9a4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass co

RuntimeError: youtube: no client resolved a stream (ERROR: [youtube] Q71sLS8h9a4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies)

## 2. Single-frame check

Confirm the stream decodes and YOLO sees the crowd before collecting anything.

In [ ]:
frame = grab_frame(stream_url)
if frame is None:
    print(f"WARN: {cam['name']} returned no frame (stream down or unreachable).")
    print('Pick another camera: change PICK in the numeric-picker cell above and re-run from there.')
else:
    print('frame shape:', frame.shape)
    print('counts:', detect_and_count(model, frame))

    res = model.predict(frame, conf=0.35, classes=[0,1,2,3,5,6,7], verbose=False)[0]
    plt.figure(figsize=(11, 6))
    plt.imshow(cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(cam['name']); plt.show()

## 3. Footfall time series (sparse sampling)

For the **how much / when** question we don't need every frame - one sample every 15-30s is plenty and
is gentle on the server. This is the same logic the collector runs continuously.

In [ ]:
def footfall_series(stream_url, cam_name, interval_s=20, duration_min=1.0):
    rows, t_end = [], time.time() + duration_min * 60
    while time.time() < t_end:
        ts = dt.datetime.now(dt.timezone.utc)
        f = grab_frame(stream_url)
        c = detect_and_count(model, f) if f is not None else {'person': np.nan, 'vehicles': np.nan}
        rows.append({'ts': ts, 'cam': cam_name, 'person': c.get('person'), 'vehicles': c.get('vehicles')})
        print(f"[{ts:%H:%M:%S}] person={c.get('person')} vehicles={c.get('vehicles')}")
        time.sleep(interval_s)
    return pd.DataFrame(rows)

# Short live-collection run. Raise duration_min for longer studies, or just leave the
# collector daemon (`python -m app.collector`) running for genuine 24/7 data.
df = footfall_series(stream_url, cam['name'], interval_s=10, duration_min=1.0)
df.to_csv(DATA_DIR / f'footfall_{CAM_ID}.csv', index=False)
df.head()

## 4. Anomalies + peak-hour profile

**Anomaly = rolling z-score > 3.5** on the footfall series: a sudden surge (event/promotion/protest) or an
unusual drop (closure/weather). Peak-hour profile tells you *when* the commercial window is.

In [ ]:
def flag_anomalies(s, window=12, z=3.5, min_delta=3):
    """Robust rolling z: median + MAD (x1.4826), the same statistic the local run
    collector uses. Outliers already inside the window inflate a mean/std
    baseline and mask the next event; a median/MAD baseline barely moves."""
    med = s.rolling(window, min_periods=4).median()
    mad = (s - med).abs().rolling(window, min_periods=4).median() * 1.4826
    spread = mad.clip(lower=1.0)   # counts are integers; floor the spread
    robust_z = (s - med) / spread
    return (robust_z.abs() > z) & ((s - med).abs() >= min_delta)

df['ts'] = pd.to_datetime(df['ts'])
df['anomaly'] = flag_anomalies(df['person'])

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(df['ts'], df['person'], marker='o', label='people')
an = df[df['anomaly'] == True]
ax[0].scatter(an['ts'], an['person'], color='red', zorder=5, label='anomaly')
ax[0].set_title('Footfall over time (robust z)'); ax[0].legend()

df['hour'] = df['ts'].dt.hour
df.groupby('hour')['person'].mean().plot(kind='bar', ax=ax[1])
ax[1].set_title('Avg people by hour (peak-hour profile)')
plt.tight_layout(); plt.show()

## 5. Dwell-time / prolonged stops (tracking)

"How long does a person or vehicle stay in front of the camera?" needs **object tracking** (stable IDs
across frames), which only works on *consecutive* frames - so here we take a short **dense burst**
(a few fps for ~60s) instead of sparse sampling. Ultralytics `model.track()` (ByteTrack) gives each
object an id; we accumulate how many frames each id is seen and how little it moves.

- **Long dwell + low movement** = lingering: window-shopping / a queue / a parked vehicle.
- High share of *lingering* people is a strong **commercial-quality** signal (people stop, not just pass).

In [ ]:
from app.detect_core import iter_frames, NAME_BY_ID

def dwell_analysis(stream_url, seconds=30, target_fps=3, conf=0.35):
    """Dense burst with tracking. Returns per-track dwell seconds + movement.

    iter_frames handles header-required hosts (tvkur, IBB, skylinewebcams) by
    downloading the latest segments with the right Referer/Origin and decoding
    locally, since cv2.VideoCapture(url) can't pass headers on Windows.
    """
    frames_seen = defaultdict(int)
    centroids = defaultdict(list)
    track_cls = {}
    n_frames = int(seconds * target_fps)
    # Sample at ~target_fps: source streams run ~25 fps, so skip frames
    # with `stride` - reading consecutive frames would compress the
    # whole window into ~n/25 seconds and overstate dwell ~8x.
    stride = max(1, round(25 / target_fps))
    for frame in iter_frames(stream_url, max_frames=n_frames, stride=stride):
        r = model.track(frame, persist=True, conf=conf, classes=[0,1,2,3,5,6,7],
                        tracker='bytetrack.yaml', verbose=False)[0]
        if r.boxes.id is not None:
            for box, tid, cl in zip(r.boxes.xywh.cpu().numpy(),
                                    r.boxes.id.int().cpu().tolist(),
                                    r.boxes.cls.int().cpu().tolist()):
                frames_seen[tid] += 1
                centroids[tid].append((float(box[0]), float(box[1])))
                track_cls[tid] = cl

    rows = []
    for tid, n in frames_seen.items():
        pts = np.array(centroids[tid])
        movement = float(np.linalg.norm(pts.max(0) - pts.min(0))) if len(pts) > 1 else 0.0
        rows.append({'track_id': tid,
                     'class': NAME_BY_ID.get(track_cls[tid], str(track_cls[tid])),
                     'dwell_s': round(n / target_fps, 1),
                     'movement_px': round(movement, 1)})
    return pd.DataFrame(rows).sort_values('dwell_s', ascending=False) if rows else pd.DataFrame(
        columns=['track_id','class','dwell_s','movement_px'])

dwell = dwell_analysis(stream_url, seconds=30, target_fps=3, conf=0.25)
dwell.head(15)

In [ ]:
# Flag prolonged stationary objects: long dwell AND little movement.
PERSON_DWELL_S, VEHICLE_DWELL_S, MAX_MOVE_PX = 25, 40, 60
if not dwell.empty:
    is_person = dwell['class'] == 'person'
    stationary = dwell[((is_person & (dwell['dwell_s'] >= PERSON_DWELL_S)) |
                        (~is_person & (dwell['dwell_s'] >= VEHICLE_DWELL_S)))
                       & (dwell['movement_px'] <= MAX_MOVE_PX)]
    print(f"Prolonged stops detected: {len(stationary)}")
    display(stationary)
    linger_rate = (is_person & (dwell['dwell_s'] >= PERSON_DWELL_S)).sum() / max(1, is_person.sum())
    print(f"Linger rate (people who stayed >= {PERSON_DWELL_S}s): {linger_rate:.0%}")

## 5b. Re-identification - "have I seen this person before?"

The detection counts above tell you *how many* people are visible at any moment, but they
double-count anyone who lingers in front of the camera. To answer questions like *"how many
unique customers walked by today?"* or *"is that the same delivery van I saw yesterday?"*
we need **re-identification**: a persistent identity attached to each person/vehicle that
survives across frames, bursts and days.

The implementation is in `app/reid.py`:

1. For each YOLO detection, crop the bounding box.
2. Build a *masked* HSV color histogram (8x8x8 bins, V<30 pixels ignored - kills the
 sodium-yellow night cast on the your picked camera square) plus aspect ratio + normalized area.
3. L2-normalize -> 514-dim appearance vector.
4. Compare to every entity of the same class already in `data/reid.db` via cosine
 similarity. If the best match is >= `threshold` (default 0.92) we update its
 `sightings` and `last_seen`; otherwise we register a new entity.

This is a **demo-grade signature**. It works well in daylight (different clothing colors
give clearly different histograms). It produces false matches at night when the whole
scene is yellow-tinted - swap `embed_crop()` for an OSNet/torchreid forward pass for
production-grade re-ID; the SQLite registry around it stays the same.

In [ ]:
if not globals().get('SELECTED_CAMS_APPLIED'):
    class _ApplyFirst(Exception):
        def _render_traceback_(self):
            return ['PAUSED: run the interactive picker cell above and pick ONE camera number first.']
    raise _ApplyFirst()
from app.detect_core import load_model, grab_frame, detect_with_boxes, annotate
from app.reid import ReidStore
import cv2, time
import matplotlib.pyplot as plt

REID_DB = str(_src_dir / 'data' / 'reid_notebook.db')
Path(REID_DB).parent.mkdir(parents=True, exist_ok=True)

# If we're re-running the notebook (the kernel is alive), the previous ReidStore is
# still holding a SQLite connection to REID_DB. Close it before we try to delete
# the file, otherwise Windows returns PermissionError [WinError 32].
try:
    reid.close()         # noqa: F821  (reid is defined by a prior run of this cell)
except NameError:
    pass

# Fresh registry for the demo so re-runs are reproducible. If something else
# still holds the file (orphan kernel, antivirus scan), we keep the existing
# rows instead of crashing - re-identification just continues with what's there.
try:
    Path(REID_DB).unlink(missing_ok=True)
    print('reid_notebook.db cleared - fresh demo registry')
except PermissionError:
    print('reid_notebook.db is locked by another process - keeping existing rows. '
          'New entities will be merged into the existing registry; this is fine '
          'for the demo, just not a clean-room run.')

reid = ReidStore(REID_DB, threshold=0.92)

# Use the model we already loaded above; lower conf so we catch the small/distant
# distant people the wide-angle camera shows.
# CAM_ID inherits from the verify cell (SELECTED_CAMS[0]) so re-ID runs on
# the same camera as the earlier sections, not a hard-coded default.
CAM_ID = SELECTED_CAMS[0]
cam = CAMERAS[CAM_ID]
stream_url = resolve_stream(cam)   # hls / skyline / webcamera24
print('feeding re-ID from', cam['name'])

In [ ]:
# Sample N frames every `interval_s` seconds, run YOLO on each, push every detection
# through the re-ID registry. Short loop here so the notebook completes; the collector
# daemon does the real long-running version.
N_SAMPLES, INTERVAL_S, CONF = 8, 5, 0.25

rows = []
for i in range(N_SAMPLES):
    f = grab_frame(stream_url)
    if f is None:
        print(f'[{i:02d}] miss'); time.sleep(INTERVAL_S); continue
    counts, boxes = detect_with_boxes(model, f, conf=CONF)
    results = reid.update_from_frame(CAM_ID, f, boxes)
    new = sum(r.is_new for r in results)
    seen_again = len(results) - new
    rows.append({'sample': i, 'person': counts['person'], 'vehicles': counts['vehicles'],
                 'detections': len(boxes), 'new_ids': new, 'seen_again': seen_again})
    print(f'[{i:02d}] person={counts["person"]} vehicles={counts["vehicles"]} '
          f'-> new={new} seen_again={seen_again}')
    time.sleep(INTERVAL_S)

reid_df = pd.DataFrame(rows)
reid_df

In [ ]:
# Roll-up: how many unique entities did we see? how many came back >=3 times?
stats = reid.stats(CAM_ID)
print('Total unique entities (this camera):', stats['total_unique'])
print('Total sightings:', stats['total_sightings'])
for cls, s in stats['per_class'].items():
    print(f"  {cls:10s}  unique={s['unique']}  sightings={s['total_sightings']}  "
          f"regulars(>=3)={s['regulars']}")

print('\nTop returning entities:')
for r in reid.top_regulars(CAM_ID, n=10):
    print(f"  #{r['entity_id']:4d}  {r['cls']:8s}  sightings={r['sightings']}  "
          f"first={r['first_seen']}  last={r['last_seen']}")

In [ ]:
# Visual: returning-visitor curve - what fraction of detections are 'seen again' over time?
if len(reid_df) >= 3:
    reid_df = reid_df.copy()
    reid_df['returning_rate'] = (reid_df['seen_again'] /
                                 reid_df['detections'].replace(0, np.nan))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(reid_df['sample'], reid_df['new_ids'], marker='o', label='new IDs')
    ax[0].plot(reid_df['sample'], reid_df['seen_again'], marker='s', label='seen again')
    ax[0].set_title('Re-ID activity per sample')
    ax[0].set_xlabel('sample #'); ax[0].set_ylabel('count'); ax[0].legend()

    ax[1].plot(reid_df['sample'], reid_df['returning_rate'].fillna(0), marker='o',
               color='#36d399')
    ax[1].set_title('Returning-visitor rate (seen_again / detections)')
    ax[1].set_xlabel('sample #'); ax[1].set_ylim(0, 1)
    plt.tight_layout(); plt.show()
else:
    print('Not enough samples for the returning-visitor plot.')

IMPORTANT - re-ID quality depends on the scene.
#
At your picked camera at night the whole scene is uniform sodium yellow.
Color-histogram re-ID will over-merge IDs there. To validate the *concept*, point
the camera at the daylight your picked camera / your picked camera (different clothing colors)
or set `threshold=0.97` to be very conservative about matches.
#
Production path:
 pip install torchreid
 from torchreid.utils import FeatureExtractor
 extractor = FeatureExtractor(model_name='osnet_ain_x1_0', model_path='', device='cpu')
 def embed_crop(crop, cls): return extractor([crop])[0].cpu().numpy()
Then keep the rest of app/reid.py exactly as-is. The 2,048-dim OSNet embedding
survives lighting changes, pose changes, and partial occlusion much better than
a color histogram.

## 6. "Is it worth opening a business here?" - a simple score

Combine three signals into one 0-100 score. Tune the weights to your business type (a cafe wants high
*linger*; a kiosk wants high *throughput*).

- **Volume** - median footfall (raw demand).
- **Linger** - share of people who stop (engagement / conversion potential).
- **Consistency** - low coefficient of variation (steady traffic beats spiky).

In [ ]:
def business_score(footfall_df, dwell_df, w=(0.5, 0.3, 0.2)):
    # Guard against an empty / all-NaN footfall series: the old
    # `if people.mean() else 1.0` fell through on NaN (NaN is truthy in
    # Python), producing NaN math that quietly clipped to 0.0 and read as
    # "bad site" instead of "insufficient data". Now we short-circuit
    # honestly.
    people = footfall_df['person'].dropna()
    if len(people) == 0:
        return {'volume_median': 0.0, 'linger_rate': 0.0,
                'consistency': 0.0, 'score_0_100': None,
                'note': 'insufficient data (no footfall samples)'}
    volume = float(people.median())
    mean = float(people.mean())
    if mean > 0:
        cv = float(people.std() / mean)
        consistency = max(0.0, 1 - cv)
    else:
        # Everyone-zero window: perfectly consistent but zero traffic - the
        # score reflects that via `vol_norm = 0`, no need to fake a cv.
        consistency = 1.0
    is_p = dwell_df['class'] == 'person'
    linger = float((is_p & (dwell_df['dwell_s'] >= 25)).sum()
                   / max(1, is_p.sum())) if len(dwell_df) else 0.0
    vol_norm = min(1.0, volume / 40.0)  # ~40 people/frame treated as 'very busy'; tune per camera FOV
    score = 100 * (w[0]*vol_norm + w[1]*linger + w[2]*consistency)
    return {'volume_median': round(volume, 1),
            'linger_rate': round(linger, 2),
            'consistency': round(consistency, 2),
            'score_0_100': round(score, 1)}

print(cam['name'])
business_score(df, dwell)


## 7. Live single-camera dashboard

The cells above ran a **local** analysis - a minute of sampling on the picked camera. The cell below embeds the live dashboard inline, showing the picked source full-width with the 10 analysis layers available (paths / pose / gestures / body / faces / line / loiter / parking / plates / heat - one at a time).

Nothing here writes to a cloud store; it's a plain local HTTP server that reads frames straight from the camera.

In [ ]:
# Serve web/ on http://localhost:PORT AND point the dashboard at the ONE camera you picked.
import json
from pathlib import Path
from app.dashboard_server import bind
from IPython.display import IFrame

PORT = 8000
_grid = {
    'country': 'local',
    'slots': [{
        'slot_id': 'local_0',
        'cam_id': SELECTED_CAM_ID,
        'display_area': SELECTED_CAM.get('area', ''),
        'placeholder_name': SELECTED_CAM['name'],
        'placeholder_hls': None,
        'placeholder_embed': None,
        'placeholder_page': SELECTED_CAM.get('url'),
        'kind': SELECTED_CAM.get('kind'),
        'path': SELECTED_CAM.get('path'),
    }]
}
_web = Path('src/web')
(_web / 'local_grid.json').write_text(json.dumps(_grid, indent=1))
print(f'local_grid.json written for cam: {SELECTED_CAM_ID}')

server = bind(PORT)
import threading
threading.Thread(target=server.serve_forever, daemon=True, name='dashboard-http').start()
import time; time.sleep(0.5)
print(f'Dashboard live at http://localhost:{PORT}/?cam={SELECTED_CAM_ID}')
IFrame(f'http://localhost:{PORT}/?cam={SELECTED_CAM_ID}', width='100%', height=780)

## 8. Compare multiple commercial sites

Loop the footfall sampler over several cameras to rank locations by activity - the input to a
site-selection decision.

In [ ]:
if not globals().get('SELECTED_CAMS_APPLIED'):
    class _ApplyFirst(Exception):
        def _render_traceback_(self):
            return ['PAUSED: run the interactive picker cell above and pick ONE camera number first.']
    raise _ApplyFirst()
# Rank the cameras YOU picked in verify7cams above (SELECTED_CAMS) by
# activity. Runs on the single camera the picker selected.
# (SELECTED_CAMS is a list-of-one for downstream-cell compatibility.)
summary = []
for cid in SELECTED_CAMS:
    c = CAMERAS.get(cid)
    if not c or not c.get('url'):
        print(f'{cid}: skipped (not in catalog or no url)')
        continue
    try:
        url = resolve_stream(c)
    except Exception as e:
        print(f'{cid}: resolve failed ({e})')
        continue
    # one quick decode check before spending 30s on this camera
    if grab_frame(url) is None:
        print(f'{cid}: no frame from stream (geo-blocked / down). Skipping.')
        continue
    sdf = footfall_series(url, c['name'], interval_s=10, duration_min=0.5)
    summary.append({'site': c['name'],
                    'median_people': sdf['person'].median(),
                    'max_people': sdf['person'].max()})

if summary:
    pd.DataFrame(summary).sort_values('median_people', ascending=False)
else:
    print('No camera in SELECTED_CAMS produced usable frames - nothing to rank.')


## 9. Live summary - what did we find?

Pulls everything the notebook saw on this run into a single block: the anomalies
flagged across every sampled camera, the re-ID totals, and a tiny visualisation
plotting all anomalies on the same timeline. Re-running the notebook regenerates
this from scratch - no stale timestamps from someone else's session leak through.

In [ ]:
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    from datetime import datetime, timezone

    print(f"Notebook run finished at {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")
    print(f"Live camera for this run: {cam['name']}")
    print("=" * 78)

    # ---- aggregate anomalies into one DataFrame ----
    anom_frames = []
    if "df" in dir() and isinstance(df, pd.DataFrame) and not df.empty and "anomaly" in df.columns:
        a = df[df["anomaly"] == True].copy()
        if not a.empty:
            a["cam"] = cam["name"]
            anom_frames.append(a[["ts", "cam", "person", "vehicles"]])
    anom = pd.concat(anom_frames, ignore_index=True) if anom_frames else \
           pd.DataFrame(columns=["ts", "cam", "person", "vehicles"])

    if len(anom):
        print(f"\nAnomalies flagged (robust rolling z > 3.5): {len(anom)}")
        print(anom.to_string(index=False))
    else:
        print("\nAnomalies flagged: 0")
        print("(Too few samples for the z-score window to trip, or the scene was steady.)")

    # ---- re-ID rollup ----
    if "reid" in dir():
        stats = reid.stats(CAM_ID)
        print("\n" + "-" * 78)
        print(f"Re-identification - {cam['name']}")
        print(f"  total unique entities   : {stats['total_unique']}")
        print(f"  total sightings         : {stats['total_sightings']}")
        for cls, s in stats["per_class"].items():
            print(f"    {cls:10s}  unique={s['unique']}  "
                  f"sightings={s['total_sightings']}  "
                  f"regulars(>=3)={s['regulars']}")
        regulars = reid.top_regulars(CAM_ID, n=5)
        if regulars:
            print("  top returning:")
            for r in regulars:
                print(f"    #{r['entity_id']:>4}  {r['cls']:8s}  "
                      f"sightings={r['sightings']}  last_seen={r['last_seen']}")

    # ---- always-on visual: footfall over this run + anomalies overlaid ----
    # Build the plot only when we have data; do NOT call ax.legend() on an empty
    # axes (that produces the "No artists with labels found" warning).
    if "df" in dir() and isinstance(df, pd.DataFrame) and not df.empty:
        ts = pd.to_datetime(df["ts"])
        fig, ax = plt.subplots(figsize=(12, 3.5))
        ax.plot(ts, df["person"],   marker="o", color="#4f8cff", label="people")
        ax.plot(ts, df["vehicles"], marker="s", color="#f0a35e", label="vehicles", alpha=0.85)
        if len(anom):
            ax.scatter(pd.to_datetime(anom["ts"]), anom["person"],
                       s=160, color="#ef4444", marker="X", zorder=5, label="anomaly")
        ax.set_title(f"This run: {cam['name']}  ({len(df)} samples, {len(anom)} anomalies)")
        ax.set_ylabel("count per frame"); ax.set_xlabel("timestamp (UTC)")
        ax.legend(loc="upper left"); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

    print("\n" + "=" * 78)
    print("For the live single-camera dashboard:")
    print("  in-notebook: run the cell in Section 7 above (embedded IFrame)")
    print("  standalone : python serve.py        (from the project root)")
    print("  open       : http://localhost:8000  (opens automatically)")

except Exception as e:
    print(f"summary cell stopped early: {type(e).__name__}: {e}")


## 10. Accuracy calibration - how good are the counts, really?

The dashboard is only as trustworthy as YOLO is on THESE cameras. This section
measures it: capture frames from the 4 live grid cameras, run the detector at
two input sizes (640 = old default, 960 = the collector's current default),
then count people/vehicles yourself and get MAE + bias per camera and per size.

Workflow (all local, ~10 minutes of counting):
1. **10a** captures frames + predictions into `data/calibration/`;
2. **10b** shows each frame - you type the true `people,vehicles`;
3. **10c** prints the accuracy table and a conf/imgsz recommendation.

Feed the result back into the pipeline: the winning `imgsz` goes to the
collector's `--imgsz`, and a camera with a systematic bias gets a `"conf"`
override in `app/cameras.py` (bias < 0 -> lower conf, bias > 0 -> raise it).

In [ ]:
if not globals().get('SELECTED_CAMS_APPLIED'):
    class _ApplyFirst(Exception):
        def _render_traceback_(self):
            return ['PAUSED: run the interactive picker cell above and pick ONE camera number first.']
    raise _ApplyFirst()
# --- 10a. Capture calibration frames + predictions (run once, ~1-2 min) ---
import json as _json
from app.detect_core import grab_burst, detect_with_boxes, annotate

CALIB_DIR = DATA_DIR / 'calibration'; CALIB_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_PER_CAM = 6          # 4 cams x 6 frames = 24 to label (aim for 20-30)
IMG_SIZES = (640, 960)      # old default vs the collector's current default
CALIB_CONF = 0.30           # keep in sync with the collector's --conf

samples = []
for cam_id in SELECTED_CAMS:
    cam = CAMERAS[cam_id]
    try:
        url = resolve_stream(cam)
    except Exception as e:
        print(f'{cam_id}: resolve failed ({e}) - skipping'); continue
    got = 0
    for k in range(FRAMES_PER_CAM):
        frames = grab_burst(url, n=1)
        if not frames:
            print(f'{cam_id}: frame {k} MISS'); continue
        frame = frames[0]
        stem = f'{cam_id}_{k:02d}'
        cv2.imwrite(str(CALIB_DIR / f'{stem}.jpg'), frame)
        entry = {'stem': stem, 'cam_id': cam_id}
        for size in IMG_SIZES:
            counts, _ = detect_with_boxes(model, frame, conf=CALIB_CONF, imgsz=size)
            entry[f'person_{size}']   = counts['person']
            entry[f'vehicles_{size}'] = counts['vehicles']
        cv2.imwrite(str(CALIB_DIR / f'{stem}_annotated.jpg'),
                    annotate(model, frame, conf=CALIB_CONF, imgsz=max(IMG_SIZES)))
        samples.append(entry); got += 1
        time.sleep(2)   # let the live stream move on a little between captures
    print(f'{cam_id}: captured {got} frames')

(CALIB_DIR / 'predictions.json').write_text(_json.dumps(samples, indent=2))
print(f'{len(samples)} frames -> {CALIB_DIR}')

In [ ]:
# INTERACTIVE cell - it asks YOU to type counts, so it must never run
# under Run All (it would stall the whole run waiting for keyboard
# input). Set RUN_LABELING = True and re-run this cell when you are
# ready to label; leave it False for hands-off runs.
RUN_LABELING = globals().get('RUN_LABELING', False)
if not RUN_LABELING:
    print('Labeling skipped (RUN_LABELING is False).')
    print('When you want to label: set RUN_LABELING = True in this cell')
    print('and run it again - it will show each frame and ask for the')
    print('true people,vehicles counts.')
else:
    # --- 10b. Label: look at each frame, type the true counts ---
    # The annotated image shows what the model saw at imgsz=960. Count what YOU
    # see: people, then vehicles (cars+buses+trucks+motorbikes+bicycles), and type
    # `people,vehicles` (e.g. `7,3`). Enter = skip frame, q = stop early.
    import json as _json

    samples = _json.loads((CALIB_DIR / 'predictions.json').read_text())
    labeled = []
    for s in samples:
        img = cv2.cvtColor(cv2.imread(str(CALIB_DIR / f"{s['stem']}_annotated.jpg")),
                           cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 7)); plt.imshow(img); plt.axis('off')
        plt.title(f"{s['stem']}  |  model@960: person={s['person_960']} "
                  f"vehicles={s['vehicles_960']}")
        plt.show()
        raw = input(f"{s['stem']} true 'people,vehicles' (Enter=skip, q=stop): ").strip()
        if raw.lower() == 'q':
            break
        if not raw:
            continue
        try:
            p_true, v_true = (int(x) for x in raw.replace(' ', '').split(','))
        except ValueError:
            print('  could not parse - skipped'); continue
        labeled.append({**s, 'person_true': p_true, 'vehicles_true': v_true})

    (CALIB_DIR / 'labeled.json').write_text(_json.dumps(labeled, indent=2))
    print(f'labeled {len(labeled)} frames -> {CALIB_DIR / "labeled.json"}')

In [ ]:
## --- 10c. Accuracy report: MAE + bias per input size and per camera ---
import json as _json

_labeled_path = CALIB_DIR / 'labeled.json'

# Fresh clone has no calibration data - degrade gracefully with a
# clear message telling the operator which cells to run first, instead
# of a raw FileNotFoundError that mentions their local user directory.
if not _labeled_path.exists():
    class _LabelFirst(Exception):
        def _render_traceback_(self):
            return [
                'PAUSED: no calibration data yet.',
                '',
                f'Expected file: {_labeled_path.name} (under src/data/calibration/).',
                '',
                'To create it, run these two cells first, in order:',
                '  10a - captures 6 frames from the picked camera at both',
                '        input sizes and drops them under src/data/calibration/.',
                '  10b - set RUN_LABELING = True and re-run it; it shows each',
                '        frame and asks you to type the true people,vehicles.',
                '',
        'Then re-run this cell. (Skip section 10 entirely if you only',
        'need the live dashboard - the calibration is optional QA.)',
            ]
    raise _LabelFirst()

try:
    rows = _json.loads(_labeled_path.read_text())
except (OSError, ValueError) as _e:
    class _LabelBad(Exception):
        def _render_traceback_(self):
            return [f'PAUSED: {_labeled_path.name} exists but is unreadable ({_e}).',
                    'Delete it and re-run cells 10a and 10b.']
    raise _LabelBad()

if not rows:
    class _LabelFirst(Exception):
        def _render_traceback_(self):
            return ['PAUSED: labeled.json is empty. Re-run cell 10b with',
                    'RUN_LABELING = True and enter at least one label.']
    raise _LabelFirst()
cal = pd.DataFrame(rows)

overall = []
for size in IMG_SIZES:
    for metric in ('person', 'vehicles'):
        err = cal[f'{metric}_{size}'] - cal[f'{metric}_true']
        overall.append({'size': size, 'metric': metric,
                        'MAE': err.abs().mean(),
                        'bias': err.mean(),
                        'n_frames': len(cal)})
print('=== Overall accuracy per input size ===')
print(pd.DataFrame(overall).to_string(index=False))

if 'cam_id' in cal.columns:
    per_cam = []
    for size in IMG_SIZES:
        for metric in ('person', 'vehicles'):
            for cid, sub in cal.groupby('cam_id'):
                err = sub[f'{metric}_{size}'] - sub[f'{metric}_true']
                per_cam.append({'cam': cid, 'size': size, 'metric': metric,
                                'MAE': err.abs().mean(),
                                'bias': err.mean(),
                                'n': len(sub)})
    print('\n=== Per-camera accuracy ===')
    print(pd.DataFrame(per_cam).to_string(index=False))
